**Innovation Management · A.Y. 2026/27 · Politecnico di Torino** · Tutorial 0, notebook 2 of 3

| | |
|---|---|
| **Goal** | Query PATSTAT with the ORM and with raw SQL, and build a first sector dataset (Step 0 of the PW). |
| **Before this** | 00 · TIP platform |
| **Time** | 30 min |
| **PATSTAT environment** | TEST (switch to PROD for final numbers) |

← [00 · The EPO Technology Intelligence Platform (TIP)](00_tip_platform.ipynb) &nbsp;|&nbsp; [02 · Step 1 of the PW: general description of the sector](02_step1_general_description.ipynb) →

# PATSTAT

PATSTAT is structured into two major sets of tables: **PATSTAT Global** and **PATSTAT EP Register**. They serve different purposes and contain different types of patent information.

**PATSTAT Global** is the main database used for most patent-data analyses. It contains bibliographic information on patent applications and publications from patent offices worldwide. Typical attributes include:

- application and publication numbers;
- filing, publication and priority dates;
- priority claims and patent-family relationships;
- applicants and inventors;
- applicant and inventor addresses and countries;
- IPC and CPC technology classifications;
- citations to patents and non-patent literature;
- information on international (PCT) applications;
- indicators related to grants and publication stages;
- links between equivalent applications filed in different patent offices.

For most empirical analyses, such as identifying technologies, constructing patent families, analysing applicants, citations, countries, or technological fields,**PATSTAT Global is usually the starting point and the most frequently used part of PATSTAT**.

> ### **PATSTAT Global contains patent data from around the world: dates, owners, inventors, technology codes, citations, and links between related applications. It’s the main database you’ll use for the project.**

**PATSTAT EP Register**, by contrast, contains detailed procedural and legal information specifically for European patent applications handled by the European Patent Office (EPO). Typical information includes:

- procedural events during examination;
- filing, search, examination and grant-related events;
- applicant and representative information recorded during the procedure;
- designated states;
- examination and opposition-related information;
- status changes and other events recorded in the European Patent Register (like if the patent is transferred, meaning if the owner changes)

The EP Register is therefore particularly useful when the research question concerns the **prosecution history or legal procedure of EP applications**, rather than only their bibliographic characteristics.

> ### **PATSTAT EP Register tracks what happens to European patent applications: examination, approval, challenges, and ownership changes. Use it when you need a patent’s legal status or history.**

The commands below display the database schemas, available tables, and relationships between them. Exploring the schema is often the fastest way to identify where a specific variable is stored and how different tables need to be joined.

For more detailed information on individual tables and variables, consult the **PATSTAT documentation/manual uploaded in the PW course material**. In particular, always check the manual when you are unsure about the precise definition of a field, its unit of observation, or the meaning of coded values.

**Contents**

1. Setup
2. Inspecting the PATSTAT database schema
3. Querying with the ORM (SQLAlchemy): four starter queries
4. Querying with raw SQL: five starter queries
5. Step 0 of the PW: data collection, an example from the literature (AI patents)
6. More advanced approaches: lexical vs semantic search (BM25 vs Patent SBERT)

---

---

> # CANNOT RUN ANYTHING AS SOME OF THE LIBRARIES ARE **ONLY** ON TIP

---

---

## 1. Setup

### Import the libraries

In [ ]:
%pip install sqlalchemy

In [ ]:
from datetime import date

import pandas as pd
from sqlalchemy import func, or_

from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import (
    TLS201_APPLN,               # applications
    TLS202_APPLN_TITLE,         # titles
    TLS203_APPLN_ABSTR,         # abstracts
    TLS206_PERSON,              # persons (applicants and inventors)
    TLS207_PERS_APPLN,          # person <-> application links
    TLS209_APPLN_IPC,           # IPC classification
    TLS224_APPLN_CPC,           # CPC classification
    TLS231_INPADOC_LEGAL_EVENT, # legal events
)

It is also possible to use

```python
from epo.tipdata.patstat.database import models
```

and then use `tab`-based auto-completion to get a hint on the list of the table models.

### Initiate the client object

**Note**

As reported by the client instantiation output below, by default you will be working with a **test database**.

It is recommended to use it during the whole development phase, as it is way faster to test. You can switch anytime to the complete database (called 'PROD') without needing to apply any change to your code.

In [ ]:
patstat = PatstatClient()          # default: env="TEST"; use PatstatClient(env="PROD") for the full database
db = patstat.orm()                 # SQLAlchemy session used by all ORM queries below

## 2. Inspecting the PATSTAT database schema

### PATSTAT Global

In [ ]:
# The schema as a picture, as provided in the official documentation
patstat.global_schema()

In [ ]:
# The same list of tables, as a Python list
patstat.list_global_tables()

# PATSTAT tables explained

These commands return the **names of the available tables**, not the data inside them:

    patstat.list_global_tables()
    patstat.list_register_tables()

Think of each table as a spreadsheet. Some hold patent details. Others connect records or explain codes.

## Basic terms

- **Application:** a request for a patent.
- **Publication:** a document published about that application.
- **Patent family:** related applications covering the same invention.
- **Applicant:** the person or organisation applying for the patent.
- **Inventor:** the person who created the invention.
- **Citation:** a reference to an earlier patent or another document.
- **ID:** a number used to identify a record and connect it to other tables.
- **Lookup table:** a table that explains what codes mean.

---

# 1. PATSTAT Global

Worldwide patent data.

Source: [EPO Global table catalogue](https://link.epo.org/web/observatory-tools/patstat/en-data-catalog-patsat-global-spring-2026.pdf).

## Applications and links between them

| Table | What it is | What it contains |
|---|---|---|
| `tls201_appln` | The main application table. Start here for basic patent data. | Application ID, number, patent office, filing date, earliest filing and publication dates, grant flag, family IDs, family size, and applicant/inventor counts. |
| `tls204_appln_prior` | Connects a later patent application to the earlier application it refers back to. Links an application to earlier applications whose filing dates it claims. This is called **priority**. Example: you first apply in Italy, then apply in Germany for the same invention. In the German application, you ask to use the Italian filing date as the invention’s starting date. That is claiming priority. | The newer application’s ID (later application’s ID), the earlier application’s ID (earlier application’s ID), and the order of the priority claims (??). |
| `tls205_tech_rel` | Links applications that EPO experts found belong together, even though the usual links are missing. Extra links between related applications identified by EPO experts, where normal priority links are missing. Example: an old French application and an old German application cover the same invention, but no official link connects them. EPO experts add that link here. | Pairs of related application IDs. **It does not measure whether two inventions are technologically similar.** |
| `tls216_appln_contn` | Links a later application to the earlier application it comes from. | Child application ID, parent application ID, and relationship type. For example, a divisional application separates part of an earlier application into another case. |

## Technology classifications

Technology codes label what an invention is about:

- **IPC:** the international classification system.
- **CPC:** a more detailed classification system.

| Table | What it is | What it contains |
|---|---|---|
| `tls209_appln_ipc` | IPC codes attached to applications. | Application ID, IPC code, classification version, and details about the code’s assignment. An application can have several codes. |
| `tls210_appln_n_cls` | Codes from national classification systems. | Application ID and national technology code. These are separate from IPC and CPC. Coverage is limited. |
| `tls222_appln_jp_class` | Japan’s own technology classifications. | Japanese application ID, classification system, and code. The systems are **FI** and **F-terms**, which add detail about technical features. |
| `tls224_appln_cpc` | CPC codes listed by application. | Application ID and CPC code. These records are derived from the family-level CPC data in `tls225`. |
| `tls225_docdb_fam_cpc` | CPC codes listed by patent family. | Family ID, CPC code, assigning office, version, assignment date, and other classification details. Several offices can assign the same code to a family. |

## Family citations and legal events

| Table | What it is | What it contains |
|---|---|---|
| `tls228_docdb_fam_citn` | Shows which patent family references another. | The citing family’s ID and the cited family’s ID. Multiple document citations between the same two families become one single family-to-family link --> Example: three patents in family B reference family A, but the table records just one B-to-A link. |
| `tls231_inpadoc_legal_event` | records official actions and changes involving a patent. | Application ID, event code, office, dates, and event details. Depending on the event, these can include fee payments, loss of protection, restoration of protection, and changes of parties. |

## Published documents and their references

| Table | What it is | What it contains |
|---|---|---|
| `tls211_pat_publn` | Published patent documents. | Publication ID, application ID, publication number, office, kind code such as A1/B1, date, language, first-grant flag, and claim count where available. **It does not contain the full claims text.** |
| `tls212_citation` | Individual citations made by patent documents. | The citing publication’s ID, the cited patent/application or non-patent document’s ID, citation origin, and related details. Use it to work out who cites whom. |
| `tls214_npl_publn` | Details of cited non-patent literature: papers, books, websites, and similar sources. | Reference text and, where available, title, authors, publication date, journal/publisher, pages, DOI, ISBN, and other details. **It does not contain the full articles.** |
| `tls215_citn_categ` | Labels explaining how a citation matters in examination. | A link to the citation, its category, and the relevant claim. For example, **X** means the cited document alone can challenge the claim’s novelty or inventive step. |

### Citation example

If patent B cites patent A:

- For **B**, this is a **backward citation**: B refers to earlier work.
- For **A**, this is a **forward citation**: later work refers to A.

## Applicants and inventors

Here, **“person” can mean a human or an organisation**.

| Table | What it is | What it contains |
|---|---|---|
| `tls206_person` | Names and addresses of applicants and inventors. | Person ID, name, address, country, region, and available standardised names such as `psn_name` and `han_name`. One real company can still appear under several IDs. |
| `tls207_pers_appln` | Connects applicants and inventors to applications. | Person ID, application ID, and applicant/inventor sequence numbers. These show the person’s role. Names and addresses come from `tls206`. |
| `tls226_person_orig` | More detailed name and address records kept in their original source form. | Original name, address parts, city, postcode, country, source, role where available, and a link to `tls206`. Useful when checking name-cleaning problems. |
| `tls227_pers_publn` | Connects applicants and inventors to particular publications. | Person ID, publication ID, and applicant/inventor sequence numbers. Unlike `tls207`, it lets you compare names recorded at different publication stages. |

## Tables that explain codes

| Table | What it is | What it contains |
|---|---|---|
| `tls801_country` | Country and patent-organisation code dictionary. | Codes, names, continent, and EU/EPO/OECD membership flags. Includes patent organisations as well as countries. |
| `tls803_legal_event_code` | Dictionary for the legal-event codes in `tls231`. | Office, event code, description, and event category. Codes need the office too, because their meanings can differ between offices. |
| `tls901_techn_field_ipc` | Connects detailed IPC codes to 35 broader technology fields. | IPC main-group code, technology-field number, field name, and broader sector. |
| `tls902_ipc_nace2` | Connects technology codes to industry categories. **NACE2** is an industry classification system. | IPC matching rules, NACE2 industry code, industry description, and weight. This mapping mainly covers manufacturing industries. |
| `tls904_nuts` | Dictionary of geographic regions. **NUTS** is the European regional coding system. | Region code, region name, and level of detail. Useful for turning an address’s region code into a readable location. |

## Applications grouped into industries and broad technology fields

| Table | What it is | What it contains |
|---|---|---|
| `tls229_appln_nace2` | assigns inventions to industries based on their technology codes. Example: an invention is linked to electronics manufacturing. This describes the invention, not necessarily the company’s business. | Application ID, NACE2 industry code, and weight. **It classifies the invention, not the applicant company’s actual business.** |
| `tls230_appln_techn_field` | groups inventions into broad technology fields. Example: an invention is grouped under telecommunications. tls901 gives you the field’s name. | Application ID, technology-field number, and weight. Use `tls901` to get the field’s name. |

A **weight** tells you how the application is divided across categories when it belongs to more than one.  
Weight is the share assigned to each category. Example: 0.5 means half the application is counted in that category.

The difference between the mapping tables and the application tables:

- `tls901` and `tls902` contain the rules connecting codes to categories.
- `tls230` and `tls229` contain the categories already assigned to individual applications.

## Patent text

| Table | What it is | What it contains |
|---|---|---|
| `tls202_appln_title` | Invention titles. | Application ID, title language, and title text. |
| `tls203_appln_abstr` | Invention abstracts—the short descriptions. | Application ID, abstract language, and abstract text. **Not the full patent description.** |

---

### PATSTAT EP Register

In [ ]:
# As a picture
patstat.register_schema()

In [ ]:
# As a Python list
patstat.list_register_tables()

# 2. PATSTAT EP Register

European patent cases and their history.

Many tables include earlier versions of records, change dates, or publication details from the **European Patent Bulletin**, the EPO’s official publication of case updates.

Source: [EPO Register table catalogue](https://link.epo.org/web/observatory-tools/patstat/en-data-catalog-patsat-ep-register-spring-2026.pdf).

## Basic case details

| Table | What it is | What it contains |
|---|---|---|
| `reg101_appln` | The main Register application table. | Register ID, application number, filing date, status code, international-application details where relevant, and `appln_id` for linking to Global. Also includes some international cases not yet in the European phase. |
| `reg102_pat_publn` | Publications connected to Register cases. | Case ID, publication office, number, kind code, date, language, and Bulletin details. |
| `reg110_title` | Titles recorded in the Register. | Case ID, title text, language, and change/publication details. |
| `reg118_prev_filed_appln` | An earlier application referred to when filing the new application **by reference**. | Case ID and the earlier application’s office, number, and filing date. This is a specific filing method, not a general family table. |
| `reg103_ipc` | IPC classifications recorded in the Register. | Case ID, IPC classification text, change date, and Bulletin details. |

## Challenges and changes to the patent

| Table | What it is | What it contains |
|---|---|---|
| `reg125_appeal` | Appeals against EPO decisions. | Case ID, appeal number, filing date, stage, dates of supporting arguments, result, and result date. |
| `reg127_petition_rvw` | Requests to review an appeal decision. | Review number, related appeal number, request date, petitioner code, decision date, and review outcome/type. |
| `reg128_limitation` | Requests to narrow what a granted patent protects. | Case ID, request date, decision information, and dates relating to acceptance, payment, or rejection. |
| `reg135_text` | Miscellaneous comments recorded about a case. | Case ID, comment text, language, and change/publication details. **It is not the patent’s full text.** |
| `reg136_search_report` | Details about searches for relevant earlier inventions and documents. | Case ID, search office, search type, date the report was sent, and the report’s publication number, date, kind, and language. **Not the full report or citation list.** |
| `reg117_relation` | Parent–child links between European applications. | Parent case ID, child case ID, relationship type, and publication details. Includes divisional applications. |
| `reg403_appln_status` | Dictionary explaining application-status codes. | Status code and readable status label. **The individual case’s status is in `reg101`; this table explains it.** |

## People, organisations, and countries

| Table | What it is | What it contains |
|---|---|---|
| `reg107_parties` | Applicants, inventors, and representatives recorded for a case. | Case ID, role, name, address, country, change date, and a flag identifying the latest set of records. |
| `reg108_applicant_states` | Countries linked to a particular applicant when applicants differ by country. | Case ID, link to the applicant record, and country. **This is not simply the applicant’s home country.** |
| `reg109_designated_states` | Countries designated in the application. | Case ID, country lists, designation type, and change details. Includes EPC member, extension, and validation states. **Designation alone does not prove current protection.** |
| `reg111_licensee` | Recorded licensees and certain other rights-holders. A licensee has permission to use the invention. | Case ID, name, address, country, licence type, date, geographic scope, and change details. **Licensing does not itself mean ownership changes.** |
| `reg112_licensee_states` | Countries covered by a recorded licence when it applies only to a subset. | Link to the licence record and the countries where that licence applies. |
| `reg130_opponent` | People or organisations formally challenging a granted patent through opposition. | Case ID, opponent’s name/address/country, representative’s details, change date, and latest-record flag. |

## Priority, steps, and events

| Table | What it is | What it contains |
|---|---|---|
| `reg106_prior` | Earlier filings whose dates the application claims as priority. | Case ID, earlier filing’s office, number, date, claim order, and changes to the priority records. |
| `reg201_proc_step` | Individual steps in handling the application. | Case ID, step ID, stage, step code, result, affected country, and time limit where relevant. |
| `reg202_proc_step_text` | Written details about those steps. | Case ID, step ID, text, and text type. Connect it to `reg201` to read more about a step. |
| `reg203_proc_step_date` | Dates attached to those steps. | Case ID, step ID, date, and date type. A step can have several dates with different meanings. |
| `reg301_event_data` | A dated log of recorded events in a case. | Case ID, event date, event code, and Bulletin details. Mainly covers the EPO process through the opposition period. |
| `reg402_event_text` | Dictionary explaining the event codes in `reg301`. | Event code and readable description. **It explains the event type; it does not list individual cases.** |
| `reg113_terms_of_grant` | Recorded losses of protection during the opposition period or opposition proceedings. | Case ID, country where protection lapsed, lapse date, and change/publication details. **Despite its name, it is a lapse table—not a list of grant conditions.** |
| `reg114_dates` | Other important dates in a case’s legal history. | Case ID, date type, date, and extra details where relevant. Examples include withdrawal, interruption/resumption of proceedings, and the end of the opposition period. |

### Steps versus events

- `reg201` gives a step’s details, results, and deadlines.
- `reg301` gives a dated event log.
- They can describe related parts of the same case.

# 3. The reg7xx tables: Unitary Patent records

A **Unitary Patent** provides one form of patent protection across participating EU countries.

These tables hold the Unitary Patent section of the Register.

**A row here does not always mean unitary protection was successfully registered. Check the status.**

| Table | What it is | What it contains |
|---|---|---|
| `reg701_appln` | The main table for cases with Unitary Patent data. | The European case ID and current Unitary Patent status code. |
| `reg705_places_of_business` | Business addresses recorded in the Unitary Patent section. | Case ID, address, country, change date, record order, and latest-record flag. |
| `reg707_parties` | Owners, inventors, and representatives in the Unitary Patent section. | Case ID, role, name, address, country, change details, and latest-record flag. |
| `reg731_event_data` | Recorded legal events for Unitary Patent cases. | Case ID, event code, event date, and Bulletin details. |
| `reg741_appln_status` | Dictionary explaining Unitary Patent status codes. | Status code and readable label. It explains the status stored in `reg701`. |
| `reg742_event_text` | Dictionary explaining Unitary Patent event codes. | Event code and readable description. It explains events stored in `reg731`. |
| `reg711_licensee` | Licensees and related recorded rights in the Unitary Patent section. | Case ID, licensee name/address/country, licence type, scope, date, and change details. |
| `reg712_licensee_states` | Countries covered by a recorded Unitary Patent licence. | Link to the licence record and country codes. |
| `reg714_dates` | Certain dates in Unitary Patent proceedings. | Case ID, date type, date, and related details. In the catalogue, these cover interruption and resumption of proceedings. |
| `reg721_proc_step` | Steps in handling Unitary Patent matters. | Case ID, step ID, stage, code, result, affected country, and time limit where relevant. |
| `reg722_proc_step_text` | Written details about those steps. | Case ID, step ID, text, and text type. Connects to `reg721`. |
| `reg723_proc_step_date` | Dates attached to those steps. | Case ID, step ID, date, and date type. Connects to `reg721`. |

# How the tables work together

Suppose you want to find **the companies filing patents in your sector**:

1. `tls224_appln_cpc` finds applications with the technology codes you chose.
2. `tls201_appln` gives their dates and family IDs.
3. `tls207_pers_appln` gives the applicant IDs.
4. `tls206_person` turns those IDs into company names.

**No single table holds the whole answer. You connect them using their IDs.**

---

## 3. Using the ORM

ORM stands for Object-Relational Mapping. It is a programming approach that allows us to interact with a relational database using Python objects and methods instead of writing SQL statements directly. The EPO provides a Python ORM interface that maps these PATSTAT tables and their columns to Python objects.
The ORM then translates the Python instructions into SQL and sends the resulting query to the PATSTAT database.

The PATSTAT client contains an implementation of **SQLAlchemy**, a well-known ORM library for Python. Conceptually, the correspondence is:

| SQL | SQLAlchemy ORM |
|---|---|
| a table | a Python model, e.g. `TLS201_APPLN` |
| a column | an attribute, e.g. `TLS201_APPLN.appln_filing_year` |
| `SELECT` | `db.query(...)` |
| `WHERE` | `.filter(...)` |
| `JOIN` | `.join(...)` |
| `ORDER BY` | `.order_by(...)` |
| `GROUP BY` | `.group_by(...)` |

In the queries below every `JOIN` states its join condition explicitly (`A.appln_id == B.appln_id`): it is more verbose, but it makes each query readable on its own and independent of the foreign keys declared in the models.

`patstat.df(query)` executes a query and returns a `pandas.DataFrame`.

### Four easy-to-understand queries to get you started with PATSTAT

#### Example 1: granted EPO applications filed in 2010

List all granted applications filed at the EPO (direct + PCT) with filing year 2010.

> Translation: Show me patent applications filed through the European Patent Office in 2010 that were eventually granted. “Direct + PCT” means include both ways of applying: directly to the EPO, or through the international application route.

In [ ]:
FILING_YEAR = 2010

q_granted_ep = (
    db.query(
        TLS201_APPLN.appln_id,
        TLS201_APPLN.appln_auth,
        TLS201_APPLN.appln_nr,
        TLS201_APPLN.appln_kind,
        TLS201_APPLN.appln_filing_date,
    )
    .filter(
        TLS201_APPLN.appln_filing_year == FILING_YEAR,
        TLS201_APPLN.appln_auth == "EP",
        TLS201_APPLN.granted == "Y",
    )
)

patstat.df(q_granted_ep)

#### Example 2: EPO applications in a time range and a specific technology field

List all applications filed at the EPO between 2010 and 2015 (included) that have a CPC classification symbol in the field of wind energy. Applications with multiple CPC classifications in wind energy will occur multiple times in the list.

Observe the blanks and the `%` wildcard in the CPC classification symbol!

In [ ]:
WIND_ENERGY_CPC = "Y02E  10/7%"   # note the two blanks: PATSTAT pads the symbol to a fixed width

q_wind_ep = (
    db.query(
        TLS201_APPLN.appln_id,
        TLS201_APPLN.appln_auth,
        TLS201_APPLN.appln_nr,
        TLS201_APPLN.appln_kind,
        TLS201_APPLN.appln_filing_date,
        TLS224_APPLN_CPC.cpc_class_symbol,
    )
    .join(TLS224_APPLN_CPC, TLS201_APPLN.appln_id == TLS224_APPLN_CPC.appln_id)
    .filter(
        TLS201_APPLN.appln_filing_year.between(2010, 2015),
        TLS201_APPLN.appln_auth == "EP",
        TLS224_APPLN_CPC.cpc_class_symbol.like(WIND_ENERGY_CPC),
    )
    .order_by(TLS201_APPLN.appln_id)
)

patstat.df(q_wind_ep)

#### Example 3: applications filed by a given applicant

List all applications filed by `NOKIA CORPORATION`.

We use the PATSTAT standardised name (`psn_name`), which gives more (better) results than the `person_name` (the name as published on the patent document).

In [ ]:
APPLICANT_PSN_NAME = "NOKIA CORPORATION"

q_nokia = (
    db.query(
        TLS201_APPLN.appln_id,
        TLS201_APPLN.appln_auth,
        TLS201_APPLN.appln_nr,
        TLS201_APPLN.appln_kind,
        TLS201_APPLN.appln_filing_date,
        TLS206_PERSON.psn_name,
        TLS202_APPLN_TITLE.appln_title,
    )
    .select_from(TLS201_APPLN)
    .join(TLS207_PERS_APPLN, TLS201_APPLN.appln_id == TLS207_PERS_APPLN.appln_id)
    .join(TLS206_PERSON, TLS207_PERS_APPLN.person_id == TLS206_PERSON.person_id)
    .join(TLS202_APPLN_TITLE, TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id)
    .filter(TLS206_PERSON.psn_name == APPLICANT_PSN_NAME)
    .order_by(TLS201_APPLN.appln_filing_date.desc())
)

patstat.df(q_nokia)

#### Example 4: a hit list of applicants

Create a hit list of the top 10 Chinese applicants filing patents at the EPO (direct or PCT).

In `TLS207_PERS_APPLN`, a person is an **applicant** when `applt_seq_nr > 0` and an **inventor** when `invt_seq_nr > 0`; the filter below keeps applicants that are not also inventors.

In [ ]:
n_applications = func.count(TLS201_APPLN.appln_id)

q_cn_applicants = (
    db.query(
        TLS206_PERSON.psn_name,
        TLS206_PERSON.person_ctry_code,
        n_applications.label("APPLICATIONS_AT_EPO"),
    )
    .select_from(TLS206_PERSON)
    .join(TLS207_PERS_APPLN, TLS206_PERSON.person_id == TLS207_PERS_APPLN.person_id)
    .join(TLS201_APPLN, TLS207_PERS_APPLN.appln_id == TLS201_APPLN.appln_id)
    .filter(
        TLS206_PERSON.person_ctry_code == "CN",
        TLS207_PERS_APPLN.applt_seq_nr > 0,   # is an applicant ...
        TLS207_PERS_APPLN.invt_seq_nr == 0,   # ... and not an inventor
        TLS201_APPLN.appln_auth == "EP",
    )
    .group_by(TLS206_PERSON.psn_name, TLS206_PERSON.person_ctry_code)
    .order_by(n_applications.desc())
    .limit(10)
)

patstat.df(q_cn_applicants)

## 4. Raw SQL

It is of course also possible to access PATSTAT directly using SQL, without the ORM layer.
The following examples replicate the previous cases, with the solution written as a direct SQL query, plus a fifth example on legal events.

`patstat.sql_query(...)` returns a list of dictionaries (one per row). Below we print the first five rows, and convert the full result to a `DataFrame` with `pd.DataFrame(rows)` when we need it.

### Five easy-to-understand SQL queries to get you started with PATSTAT

#### Example 1

List all granted applications filed at the EPO (direct + PCT) with filing year 2010.

**Note** This and the next example use the **legacy SQL** dialect, which, for example, allows the `+` notation in the `SELECT` statement (see `appln_auth + appln_nr + appln_kind AS number` below).

The legacy SQL dialect is selected by *default* for backwards compatibility with legacy PATSTAT scripts. It is controlled by the parameter `use_legacy_sql` (default `True`).

In [ ]:
sql_granted_ep = """
SELECT appln_id,
       appln_auth + appln_nr + appln_kind AS number,
       appln_filing_date
FROM   tls201_appln
WHERE  appln_filing_year = 2010   -- application filing year = 2010
  AND  appln_auth = 'EP'          -- filed at the EPO (direct + PCT)
  AND  granted = 'Y'              -- the patent has been granted
"""

rows = patstat.sql_query(sql_granted_ep)
rows[:5]

#### Example 2

List all applications filed at the EPO (direct + PCT) between 2010 and 2015 (included) that have a CPC classification symbol in the field of wind energy. Applications with multiple CPC classifications in wind energy will occur multiple times in the list.

Observe the blanks and the `%` wildcard in the CPC classification symbol!

In [ ]:
sql_wind_ep = """
SELECT tls201_appln.appln_id,
       appln_auth + appln_nr + appln_kind AS number,
       appln_filing_date,
       cpc_class_symbol
FROM   tls201_appln
JOIN   tls224_appln_cpc ON tls201_appln.appln_id = tls224_appln_cpc.appln_id
WHERE  appln_filing_year BETWEEN 2010 AND 2015   -- filed between 2010 and 2015
  AND  appln_auth = 'EP'                         -- filed at the EPO (direct + PCT)
  AND  cpc_class_symbol LIKE 'Y02E  10/7%'       -- CPC class for wind energy
ORDER BY tls201_appln.appln_id                   -- one block of rows per application
"""

rows = patstat.sql_query(sql_wind_ep)
rows[:5]

#### Example 3

List all applications filed by `NOKIA CORPORATION`. We use the PATSTAT standardised name (`psn_name`), which gives more (better) results than the `person_name` (the name as published on the patent document).

**Note** In this and the following examples we use the **standard SQL** dialect (parameter `use_legacy_sql=False`).

**Note 2** It is **not** necessary to enter the name of the project and the dataset in the query string (which standard BigQuery SQL normally requires). The `sql_query` method does this for you!

In [ ]:
sql_nokia = """
SELECT tls201_appln.appln_id,
       appln_auth,
       appln_nr,
       appln_kind,
       appln_filing_date,
       psn_name,
       appln_nr_epodoc,
       appln_title
FROM   tls201_appln
JOIN   tls207_pers_appln  ON tls201_appln.appln_id      = tls207_pers_appln.appln_id
JOIN   tls206_person      ON tls207_pers_appln.person_id = tls206_person.person_id
JOIN   tls202_appln_title ON tls202_appln_title.appln_id = tls201_appln.appln_id
WHERE  psn_name LIKE 'NOKIA CORPORATION'   -- all applications filed by NOKIA CORPORATION
ORDER BY appln_filing_date DESC            -- most recent first
"""

rows = patstat.sql_query(sql_nokia, use_legacy_sql=False)
rows[:5]

#### Example 4

Create a hit list of Chinese applicants filing patents at the EPO (direct or PCT).

In [ ]:
sql_cn_applicants = """
SELECT psn_name,
       person_ctry_code,
       COUNT(tls201_appln.appln_id) AS APPLICATIONS_AT_EPO     -- number of applications
FROM   tls201_appln
JOIN   tls207_pers_appln ON tls201_appln.appln_id      = tls207_pers_appln.appln_id
JOIN   tls206_person     ON tls207_pers_appln.person_id = tls206_person.person_id
WHERE  person_ctry_code = 'CN'                    -- Chinese "persons"
  AND  applt_seq_nr > 0 AND invt_seq_nr = 0       -- applicants only, exclude inventors
  AND  appln_auth = 'EP'                          -- filed at the EPO
GROUP BY psn_name, person_ctry_code               -- one total per name/country pair
ORDER BY COUNT(tls201_appln.appln_id) DESC        -- most applications first
"""

rows = patstat.sql_query(sql_cn_applicants, use_legacy_sql=False)
rows[:5]

#### Example 5

List the granted EP applications of Chinese applicants for which a renewal fee was paid in Belgium, using the INPADOC legal events table (`tls231_inpadoc_legal_event`).

The event code `PGFP` means that a post-grant (renewal) fee was paid, which implies the EP application was granted.

In [ ]:
sql_cn_fees_be = """
SELECT DISTINCT psn_name,
       person_ctry_code,
       appln_auth,
       appln_nr,
       appln_kind,
       appln_filing_date,
       fee_country,
       fee_payment_date,
       fee_renewal_year,
       appln_title,
       tls201_appln.appln_id
FROM   tls201_appln
JOIN   tls207_pers_appln          ON tls201_appln.appln_id      = tls207_pers_appln.appln_id
JOIN   tls206_person              ON tls207_pers_appln.person_id = tls206_person.person_id
JOIN   tls231_inpadoc_legal_event ON tls201_appln.appln_id      = tls231_inpadoc_legal_event.appln_id
JOIN   tls202_appln_title         ON tls201_appln.appln_id      = tls202_appln_title.appln_id
WHERE  person_ctry_code = 'CN'    -- Chinese "persons"
  AND  applt_seq_nr > 0           -- applicants only
  AND  appln_auth = 'EP'          -- filed at the EPO
  AND  event_code = 'PGFP'        -- a post-grant fee was paid (implies a granted EP)
  AND  fee_country = 'BE'         -- fee paid in Belgium
ORDER BY fee_payment_date DESC    -- most recent payment first
"""

rows = patstat.sql_query(sql_cn_fees_be, use_legacy_sql=False)
rows[:5]

# 5. Step 0 of the PW: data collection, an example

### Example from a paper (uploaded in Materials > academic_papers)
## Mapping technological innovation dynamics in artificial intelligence domains: Evidence from a global patent analysis
Authors: Na Liu, Philip Shapira, Xiaoxu Yue, Jiancheng Guan

### Goal: identify patents related to the field of AI
### Strategy: combine IPC/CPC codes with keywords in selected text fields

The search is built in three independent blocks (keywords, IPC codes, CPC codes). Each block is a reusable function that returns the `appln_id`s it selects; the final AI set is their union:

$$\text{AI} = \text{Keywords} \cup \text{IPC} \cup \text{CPC}$$

### Original query

![tip](img/query_ai.png)

### 5.1 Search definitions

The three lists below are the *data* of the search strategy, kept separate from the *code* that runs it: to adapt the strategy to your sector you only change these constants.

In [ ]:
# Keywords searched (case-insensitive) in titles and abstracts
AI_KEYWORDS = (
    "artificial intelligence",
    "neural net",
    "machine learning",
    "expert system",
    "natural language processing",
    "deep learning",
    "reinforcement learning",
    "reinforced learning",
    "learning algorithm",
    "supervised learning",
    "intelligent agent",
    "back propagation learning",
    "BP learning",
    "back propagation algorithm",
    "long short-term memory",
    "LSTM",
    "pulse coupled neural net",
    "perceptron",
    "neuro evolution",
    "liquid state machine",
    "deep belief net",
    "radial basis function net",
    "RBFNN",
    "RBF net",
    "deep net",
    "autoencoder",
    "committee machine",
    "training algorithm",
    "back propagation net",
    "BP network",
    "Q learning",
    "convolution net",
    "actor-critic algorithm",
    "feed forward net",
    "hopfield net",
    "neocognitron",
    "xgboost",
    "boltzmann machine",
    "activation function",
    "neuro dynamic programming",
    "learning model",
    "neuro computing",
    "temporal difference learning",
    "echo state net",
    "transfer learning",
    "gradient boosting",
    "adversarial learning",
    "feature learning",
    "generative adversarial net",
    "representation learning",
    "multi agent learning",
    "reservoir computing",
    "co-training",
    "probably approximate correct learning",
    "extreme learning machine",
    "ensemble learning",
    "machine intelligence",
    "neuro fuzzy",
    "lazy learning",
    "multi instance learning",
    "multi task learning",
    "computational intelligence",
    "neural model",
    "multi label learning",
    "similarity learning",
    "statistical relation learning",
    "support vector regression",
    "manifold regularization",
    "decision forest",
    "generalization error",
    "transductive learning",
    "neuro robotic",
    "inductive logic programming",
    "natural language understanding",
    "adaboost",
    "adaptive boosting",
    "incremental learning",
    "random forest",
    "metric learning",
    "neural gas",
    "grammatical inference",
    "support vector machine",
    "multi label classification",
    "conditional random field",
    "multi class classification",
    "mixture of expert",
    "concept drift",
    "genetic programming",
    "string kernel",
    "learning to rank",
    "machine-learned ranking",
    "boosting algorithm",
    "robot learning",
    "relevance vector machine",
    "connectionism",
    "multi kernel learning",
    "graph learning",
    "naive bayes classifier",
    "rule-based system",
    "classification algorithm",
    "graph kernel",
    "rule induction",
    "manifold learning",
    "label propagation",
    "hypergraph learning",
    "one class classifier",
    "intelligent algorithm",
)

In [ ]:
# IPC codes: every symbol starting with one of these prefixes is selected
AI_IPC_CODES = (
    "A63F13/67",
    "G06F8/33",
    "G06F15/18",
    "G06F17/20",
    "G06F17/21",
    "G06F17/27",
    "G06F17/28",
    "G06F19/24",
    "G06K9/66",
    "G06N3/02",
    "G06N3/04",
    "G06N3/06",
    "G06N3/063",
    "G06N3/067",
    "G06N3/08",
    "G06N3/10",
    "G06N5/00",
    "G06N5/02",
    "G06N5/04",
    "G06N7/02",
    "G06N20",
    "G06N20/00",
    "G06N20/10",
    "G06N20/20",
    "G06T1/40",
    "G10L15/06",
    "G10L15/16",
    "G10L15/18",
    "G10L17/04",
    "G10L17/10",
    "G10L17/18",
    "G10L25/30",
    "G16B40",
    "G16B40/00",
    "G16B40/20",
    "G16B40/30",
    "G16C20/70",
    "G16H50/20",
    "H01M8/04992",
)

In [ ]:
# CPC codes: every symbol starting with one of these prefixes is selected
AI_CPC_CODES = (
    "A61B5/7264",
    "A61B5/7267",
    "A63F13/67",
    "B23K31/006",
    "B25J9/161",
    "B25J9/163",
    "B29C66/965",
    "B29C2945/76946",
    "B29C2945/76949",
    "B29C2945/76979",
    "Y10S128/924",
    "Y10S128/925",
    "Y10S706",
)

### 5.2 Reusable query builders

Two details matter here:

- **Keywords** are matched with `LIKE '%term%'` on the lower-cased title and abstract, so `"neural net"` also matches *neural network(s)*.
- **Classification symbols** are stored in PATSTAT with padding blanks (e.g. `G06N   3/08`). We remove the blanks on both sides before comparing, and match by prefix (`LIKE 'G06N3/08%'`), so a code also selects all its subgroups.

In [ ]:
# terms is the list of keywords / IPC / CPC codes we want to look for
def keyword_condition(terms):
    '''
    We pass this function the whole list of keywords / IPC codes / CPC codes we're looking for
    For each of these words build a condition that checks:
        - is the considered word in the title? Append check in list
        - is the considered word in the abstract? Append check in list
    So at the end of the loop the conditions list will contain: [does term 1 appear in title? does term 1 appear in abstract? does term 2 appear in title? does term 2 appear in abstract? ...]
    The final condition combines all these conditions with OR --> [does term 1 appear in title OR does term 1 appear in abstract? OR does term 2 appear in title OR does term 2 appear in abstract? OR ...]
    It returns both titles and abstracts when either the title or the abstract contains at least one keyword.

    The idea is to use this condition like: query = query.filter(combined_condition)
    '''

    lowercase_title = func.lower(TLS202_APPLN_TITLE.appln_title)    # title is in table TLS202

    lowercase_abstract = func.lower(TLS203_APPLN_ABSTR.appln_abstract)  # abstract is in table TLS203

    conditions = []

    for term in terms:
        lowercase_term = term.lower()
        search_pattern = "%" + lowercase_term + "%"     # example "%robot%" --> it finds "robot" anywhere in the text: "robot", "robotics", or "a new robot arm".

        title_condition = lowercase_title.like(search_pattern)          # check whether the title contains ‘robot’ --> THIS IS AN INSTRUCTION
        abstract_condition = lowercase_abstract.like(search_pattern)    # check whether the abstract contains ‘robot’ --> THIS IS AN INSTRUCTION

        # puts both instructions into a Python list
        conditions.append(title_condition)
        conditions.append(abstract_condition)

    # the condition is: the title contains robot OR the abstract contains robot
    combined_condition = or_(*conditions)
    
    # it's like:
    # combined_condition = or_(
    #     title_condition,
    #     abstract_condition
    # )

    return combined_condition



def symbol_prefix_condition(symbol_column, codes):
    '''
    symbol_column = the database column containing the technology codes you want to check. Example: TLS224_APPLN_CPC.cpc_class_symbol (CPC code)
    codes = IPC or CPC codes

    Use this to search for IPC or CPC codes.
    A code matches if it starts with ANY prefix in codes.

    Steps:
    1. Remove spaces from the database's code.
    2. Remove spaces from each search prefix.
    3. Add "%" at the end, meaning anything can follow.
    4. Create a check for each prefix.
    5. Combine the checks with OR.

    Removing spaces avoids problems caused by PATSTAT's spacing.
    For example, "Y02E  10/7" becomes "Y02E10/7".

    This returns a filter condition. It does not run a search.
    '''

    symbol_without_spaces = func.replace(
        symbol_column,
        " ",
        ""
    )

    conditions = []

    for code in codes:
        prefix_without_spaces = code.replace(" ", "")
        search_pattern = prefix_without_spaces + "%"        # search fo that prefix and it can be folled by anything but it has to start with the prefix

        code_condition = symbol_without_spaces.like(search_pattern)     # build the condition that checks for codes starting with the considered code

        conditions.append(code_condition)       # append the condition into a list --> list of conditions: [check whether symbol column starts with code prefix1, check whether symbol column starts with code prefix2, ...]

    combined_condition = or_(*conditions)   # combine all conditions with OR

    return combined_condition


# These are the columns we want to see in the results.
# They do not decide which applications match.
APPLN_COLUMNS = (
    TLS201_APPLN.appln_id,
    TLS201_APPLN.appln_auth,
    TLS201_APPLN.appln_nr,
    TLS201_APPLN.appln_kind,
    TLS201_APPLN.appln_filing_date,
)

# NOTE
# placing an asterisk in the input function = accept any number of arguments and collects them into a tuple called columns.
# def example(*columns):
#     print(columns)
# example("title", "abstract")

# Prints:
# ("title", "abstract")

def query_by_keywords(*columns):
    '''
    columns = the columns we want to extract from the database and show in the output

    We need to join the table TLS201 with table TLS202 because it contains the application title
    We need to join the table TLS201 with table TLS203 because it contains the application abstract
    
    Use this to find applications mentioning an AI keyword.

    AI_KEYWORDS must already contain your list of search words.
    columns tells the query which columns to show.

    Steps:
    1. Choose the output columns.
    2. Start from the application table.
    3. Connect each application to its titles and abstracts
       using the application ID.
    4. Keep applications where a title or abstract contains
       at least one word from AI_KEYWORDS.

    An outer join keeps an application even if one text field
    is missing. For example, it can match through its title
    even when it has no abstract.

    This builds the query. Use patstat.df(query) to run it.
    '''

    # columns we want to select in the output --> SELECT
    query = db.query(*columns)

    # FROM
    query = query.select_from(TLS201_APPLN)

    # WHERE (join conditions) TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id AND TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id

    query = query.outerjoin(
        TLS202_APPLN_TITLE,
        TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id    # title link
    )

    query = query.outerjoin(
        TLS203_APPLN_ABSTR,
        TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id    # abstract link
    )

    # recall function to create the keywords condition that checks for any of the keywords in title OR abstract
    matching_keywords = keyword_condition(AI_KEYWORDS)

    # applies the keywords conditions
    query = query.filter(matching_keywords)

    return query



def query_by_ipc(*columns):
    '''
    Use this to find applications through their IPC codes.

    AI_IPC_CODES must already contain your list of AI code prefixes.
    columns tells the query which columns to show.

    Steps:
    1. Choose the output columns.
    2. Start from the application table.
    3. Connect each application to its IPC codes
       using the application ID.
    4. Keep codes starting with at least one prefix
       from AI_IPC_CODES.

    A normal join leaves out applications with no IPC records.
    One application can appear several times if several codes match.

    This builds the query. Use patstat.df(query) to run it.
    '''

    query = db.query(*columns)

    query = query.select_from(TLS201_APPLN)


    query = query.join(
        TLS209_APPLN_IPC,
        TLS201_APPLN.appln_id == TLS209_APPLN_IPC.appln_id
    )

    # we recall the function (we pass the IPC column + the IPC codes list we're looking for) to build the condition that looks for records containing ANY of the IPC codes
    matching_ipc_codes = symbol_prefix_condition(
        TLS209_APPLN_IPC.ipc_class_symbol,
        AI_IPC_CODES
    )

    query = query.filter(matching_ipc_codes)

    return query



def query_by_cpc(*columns):
    '''
    Use this to find applications through their CPC codes.

    AI_CPC_CODES must already contain your list of AI code prefixes.
    columns tells the query which columns to show.

    Steps:
    1. Choose the output columns.
    2. Start from the application table.
    3. Connect each application to its CPC codes
       using the application ID.
    4. Keep codes starting with at least one prefix
       from AI_CPC_CODES.

    A normal join leaves out applications with no CPC records.
    One application can appear several times if several codes match.

    This builds the query. Use patstat.df(query) to run it.
    '''

    query = db.query(*columns)

    query = query.select_from(TLS201_APPLN)

    query = query.join(
        TLS224_APPLN_CPC,
        TLS201_APPLN.appln_id == TLS224_APPLN_CPC.appln_id      # CPC link
    )

    # we recall the function (we pass the CPC column + the CPC codes list we're looking for) to build the condition that looks for records containing ANY of the CPC codes
    matching_cpc_codes = symbol_prefix_condition(
        TLS224_APPLN_CPC.cpc_class_symbol,
        AI_CPC_CODES
    )

    query = query.filter(matching_cpc_codes)

    return query

### 5.3 Query 1: keyword search

Search AI-related terms in patent titles and abstracts.

In [ ]:
q_keywords = query_by_keywords(
    *APPLN_COLUMNS,
    TLS202_APPLN_TITLE.appln_title,
    TLS203_APPLN_ABSTR.appln_abstract,
).distinct()

df_keywords = patstat.df(q_keywords)
df_keywords.head(10)

### 5.4 Query 2: IPC search

In [ ]:
q_ipc = query_by_ipc(*APPLN_COLUMNS, TLS209_APPLN_IPC.ipc_class_symbol).distinct()

df_ipc = patstat.df(q_ipc)
df_ipc.head()

### 5.5 Query 3: CPC search

In [ ]:
q_cpc = query_by_cpc(*APPLN_COLUMNS, TLS224_APPLN_CPC.cpc_class_symbol).distinct()

df_cpc = patstat.df(q_cpc)
df_cpc.head(10)

### 5.6 Query 4: combine the three search strategies

AI patent = keyword search **OR** IPC search **OR** CPC search.

In SQL an OR across different tables is a `UNION` of the three sets of `appln_id`s (`UNION` also removes duplicates). We reuse the same builders, now selecting only the `appln_id`, and then join the union back to the applications to retrieve their details.


### Part 1: combine the three searches
The code performs the searches --> imagine the searches return:
```text
Keyword search:  10, 20
IPC search:      20, 30
CPC search:      30, 40
```

UNION combines them and removes repeated IDs:  
10, 20, 30, 40

An application only needs to match one search to enter this list.


### Part 2: get the details for those IDs
Now we have the IDs of those applications respecting EITHER one of the conditions --> query the resulting IDs to decide what to show

In [ ]:
# useless, it renames the column with the name it already has --> the professor is a bot
appln_id = TLS201_APPLN.appln_id.label("appln_id")

# this contains the IDs of all records that respect at least one of the conditions:
    # - either the title OR the abstract contain at least one of the keywords
    # - the IPC code is one of the searched IPC values
    # - the CPC code is one of the searched CPC values
ai_ids = (
    query_by_keywords(appln_id)
    .union(query_by_ipc(appln_id), query_by_cpc(appln_id))
    .subquery() # subquery allows to use the results of this query for another query --> we can query the results inside ai_ids
)


q_ai = (
    db.query(*APPLN_COLUMNS, TLS202_APPLN_TITLE.appln_title)    # choose what to show --> SELECT
    
    .select_from(TLS201_APPLN)                                  # FROM
    
    .join(ai_ids, TLS201_APPLN.appln_id == ai_ids.c.appln_id)   # WHERE --> match applications to the IDs found by our three searches.

    .outerjoin(TLS202_APPLN_TITLE, TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id)

    .distinct()                                                 # Remove identical output rows.

    .order_by(TLS201_APPLN.appln_filing_date)                   # Sort by filing date, oldest first.
)


df_ai = patstat.df(q_ai)
df_ai.head(10)

In [ ]:
# How much does each strategy contribute? (counts of distinct applications)
pd.Series(
    {
        "keywords": df_keywords["appln_id"].nunique(),  # count the different applications found by the keyword search, and label that count ‘keywords’
        # apparently df_keywords = the results table from the keyword search
        # .nunique() = counts different IDs --> repeated IDs count only once

        "IPC": df_ipc["appln_id"].nunique(),
        "CPC": df_cpc["appln_id"].nunique(),
        "union (AI set)": df_ai["appln_id"].nunique(),
    },
    name="applications",
)

# 6. More advanced approaches (ideas)
##### Remember: adapt the technique to your specific task

### Advanced technology identification: lexical vs semantic search

Patent classifications provide a useful starting point for defining a broad
technological domain, but they may be too broad to identify a narrower
technology of interest.

In this example, we use CPC **Y02C20/40 – Capture or disposal of CO₂**
as the candidate technological space and investigate whether patents related
specifically to **Direct Air Capture (DAC)** can be identified from their
abstracts.

We compare two retrieval approaches:

1. **BM25**, a traditional lexical information-retrieval method that ranks
   documents according to the occurrence and informativeness of query terms;

2. **Patent SBERT-adapt-ub**, a domain-adapted SentenceTransformer model that
   represents patent texts as dense semantic vectors and ranks patents according
   to cosine similarity with a textual description of the technology.

Using the same set of patent abstracts allows us to compare lexical similarity
with semantic similarity directly.

Patent SBERT-adapt-ub is introduced in:

Ascione, G. S., & Sterzi, V. (2024).
*A comparative analysis of embedding models for patent similarity*.
arXiv:2403.16630.

Model:
https://huggingface.co/graziasveva93/sbert-adapt-ub

> ### Less words, more substance

### Finding DAC patents: words vs meaning
CPC Y02C20/40 covers CO₂ capture and disposal. We want to find a smaller group: Direct Air Capture (DAC)—removing CO₂ directly from the air.  

We search the same patent abstracts in two ways:
- BM25: looks for matching words.
- Patent SBERT: looks for similar meaning, even when different words are used.

Then we compare the results.

### 6.0 Install and import the NLP libraries

In [ ]:
%pip install -q rank-bm25 sentence-transformers umap-learn

In [ ]:
import re

import matplotlib.pyplot as plt
import numpy as np
import umap
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

### 6.1 Configuration



In [ ]:
CANDIDATE_CPC = "Y02C  20/40"       # Capture or disposal of CO2 (note the padding blanks)
MIN_FILING_DATE = date(2005, 1, 1)  # patents filed from 2005 onwards
ABSTRACT_LANGUAGE = "en"

SBERT_MODEL = "graziasveva93/sbert-adapt-ub"
TOP_N_DISPLAY = 5                   # rows shown in the ranking tables
TOP_K = 100                         # size of the top lists compared in the overlap and the maps
RANDOM_STATE = 42                   # for a reproducible UMAP projection

DAC_DESCRIPTION = """
Technology for removing carbon dioxide directly from ambient atmospheric air,
where carbon dioxide is present at low concentration. The process may use
solid sorbents, liquid solvents, membranes, adsorption, absorption or other
selective separation techniques to capture CO2 from air, followed by
regeneration of the capture material and release of concentrated carbon
dioxide for storage or utilisation. The technology differs from point-source
carbon capture because the carbon dioxide is extracted directly from the
atmosphere rather than from concentrated industrial flue gases.
"""

### 6.2 Candidate technological space

CPC Y02C20/40, patents filed from 2005 onwards, with an English abstract.

**NB** Remember: in this example we are using TEST, so we are not getting all patents.

In [ ]:
q_candidates = (
    db.query(
        *APPLN_COLUMNS,
        TLS202_APPLN_TITLE.appln_title,
        TLS203_APPLN_ABSTR.appln_abstract,
        TLS224_APPLN_CPC.cpc_class_symbol,
    )
    .select_from(TLS201_APPLN)
    .join(TLS224_APPLN_CPC, TLS201_APPLN.appln_id == TLS224_APPLN_CPC.appln_id)
    .join(TLS203_APPLN_ABSTR, TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id)
    .outerjoin(TLS202_APPLN_TITLE, TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id)
    .filter(
        TLS224_APPLN_CPC.cpc_class_symbol == CANDIDATE_CPC,
        TLS201_APPLN.appln_filing_date >= MIN_FILING_DATE,
        TLS203_APPLN_ABSTR.appln_abstract.isnot(None),
        TLS203_APPLN_ABSTR.appln_abstract_lg == ABSTRACT_LANGUAGE,
    )
    .distinct()
)

candidates = (
    patstat.df(q_candidates)
    .dropna(subset=["appln_abstract"])
    .drop_duplicates(subset=["appln_id"])   # one row per application
    .reset_index(drop=True)
)

print(f"Candidate patents: {len(candidates):,}")
candidates

### 6.3 Shared helpers

Both methods produce one score per abstract. The helpers below turn a score into a ranking and display the top results in the same format, so the two methods are compared on equal terms.

In [ ]:
def rank_by(df, score_col):
    """Return a copy of df sorted by score_col (descending) with a 1-based rank column."""
    ranked = df.sort_values(score_col, ascending=False).reset_index(drop=True)
    ranked.insert(0, "rank", np.arange(1, len(ranked) + 1))
    return ranked


def top_table(ranked, score_col, n=TOP_N_DISPLAY):
    """The n best-ranked applications, with the columns needed to judge them."""
    cols = ["rank", "appln_id", "appln_title", "appln_filing_date", score_col, "appln_abstract"]
    return ranked[cols].head(n)


def top_k_ids(ranked, k=TOP_K):
    """The set of appln_ids among the k best-ranked applications."""
    return set(ranked["appln_id"].head(k))

### 6.4 Method 1: BM25 (lexical)

Abstracts and the technology description are tokenised into lower-case alphanumeric words; BM25 then scores each abstract by the query words it contains, weighting rare words more.

In [ ]:
TOKEN_PATTERN = re.compile(r"\b[a-zA-Z0-9]+\b")


def tokenize(text):
    """Lower-case alphanumeric tokens."""
    return TOKEN_PATTERN.findall(text.lower())


bm25 = BM25Okapi([tokenize(text) for text in candidates["appln_abstract"].astype(str)])
candidates["bm25_score"] = bm25.get_scores(tokenize(DAC_DESCRIPTION))

ranked_bm25 = rank_by(candidates, "bm25_score")
top_table(ranked_bm25, "bm25_score")

### 6.5 Method 2: Patent SBERT-adapt-ub (semantic)

Each abstract and the technology description are encoded as dense vectors. With `normalize_embeddings=True` every vector has unit length, so the dot product is exactly the **cosine similarity**.

In [ ]:
model = SentenceTransformer(SBERT_MODEL)

abstract_embeddings = model.encode(
    candidates["appln_abstract"].astype(str).tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)
dac_embedding = model.encode(DAC_DESCRIPTION, convert_to_numpy=True, normalize_embeddings=True)

print("Embedding matrix:", abstract_embeddings.shape)   # (n_patents, embedding_dim)

In [ ]:
candidates["sbert_similarity"] = abstract_embeddings @ dac_embedding   # cosine similarity

ranked_sbert = rank_by(candidates, "sbert_similarity")
top_table(ranked_sbert, "sbert_similarity")

### 6.6 Comparing the two rankings

In [ ]:
n = TOP_N_DISPLAY
comparison_top = pd.DataFrame(
    {
        "rank": range(1, n + 1),
        "BM25 appln_id": ranked_bm25["appln_id"].head(n).values,
        "BM25 title": ranked_bm25["appln_title"].head(n).values,
        "SBERT appln_id": ranked_sbert["appln_id"].head(n).values,
        "SBERT title": ranked_sbert["appln_title"].head(n).values,
    }
)
comparison_top

How many patents do the two methods agree on among their top *K* (`TOP_K`) results?

**Caution** In TEST the candidate set is small: when `TOP_K` is close to the number of candidates, the two top lists overlap almost by construction. Interpret the overlap with PROD data, or with a smaller `TOP_K`.

In [ ]:
k = min(TOP_K, len(candidates))   # cannot take more patents than there are candidates

bm25_top_ids = top_k_ids(ranked_bm25, k)
sbert_top_ids = top_k_ids(ranked_sbert, k)
overlap = bm25_top_ids & sbert_top_ids

print(f"Candidates: {len(candidates)} | top-K: {k}")
print(f"Patents appearing in both top-{k} lists: {len(overlap)}")
print(f"Top-{k} overlap: {len(overlap) / k:.1%}")

candidates["in_bm25_top"] = candidates["appln_id"].isin(bm25_top_ids)
candidates["in_sbert_top"] = candidates["appln_id"].isin(sbert_top_ids)

### 6.7 Visualising the semantic space with UMAP

UMAP projects the embeddings to 2 dimensions. The technology description is projected **together with** the abstracts (it is the last row of the input matrix), so it can be shown on the same map.

In [ ]:
umap_input = np.vstack([abstract_embeddings, dac_embedding.reshape(1, -1)])

reducer = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    metric="cosine",
    random_state=RANDOM_STATE,
)
coords = reducer.fit_transform(umap_input)

candidates[["UMAP_1", "UMAP_2"]] = coords[:-1]   # one point per patent
dac_xy = coords[-1]                               # the technology description

In [ ]:
def plot_retrieval_map(df, highlight_col, highlight_label, title, query_xy=dac_xy):
    """UMAP map of all candidates, highlighting the patents selected by one method."""
    selected = df[df[highlight_col]]
    background = df[~df[highlight_col]]

    fig, ax = plt.subplots(figsize=(12, 8))
    ax.scatter(background["UMAP_1"], background["UMAP_2"], s=10, alpha=0.15,
               label="Other Y02C20/40 patents")
    ax.scatter(selected["UMAP_1"], selected["UMAP_2"], s=45, alpha=0.8,
               label=highlight_label)
    ax.scatter(*query_xy, marker="*", s=350, label="Direct Air Capture description")

    ax.set_xlabel("UMAP dimension 1")
    ax.set_ylabel("UMAP dimension 2")
    ax.set_title(title)
    ax.legend()
    plt.show()

In [ ]:
plot_retrieval_map(
    candidates,
    highlight_col="in_bm25_top",
    highlight_label=f"BM25 top {k}",
    title="BM25 retrieval of Direct Air Capture patents\nwithin CPC Y02C20/40",
)

In [ ]:
plot_retrieval_map(
    candidates,
    highlight_col="in_sbert_top",
    highlight_label=f"Patent SBERT-adapt-ub top {k}",
    title="Semantic retrieval of Direct Air Capture patents\nwithin CPC Y02C20/40",
)

---

← [00 · The EPO Technology Intelligence Platform (TIP)](00_tip_platform.ipynb) &nbsp;|&nbsp; [Back to the README](../README.md) &nbsp;|&nbsp; [02 · Step 1 of the PW: general description of the sector](02_step1_general_description.ipynb) →